> **Portfolio summary:** Baseline RAG system: dense vector search over 20 UK fiscal-policy publications, used as the comparison point for the enhanced version.
>
> [Project README](README.md) · [Back to the portfolio](../README.md)

# Baseline RAG Pipeline

This notebook implements a baseline (vanilla) Retrieval-Augmented Generation (RAG) pipeline. The objective is to demonstrate a simple end-to-end system that retrieves relevant information from a document corpus and generates responses using a language model.

This baseline serves as a benchmark system, against which the enhanced RAG pipeline will be evaluated.


# Pipeline Structure Overview

The following outlines the structure of the baseline RAG pipeline:

Data ingestion - Text preprocessing - Chunking - Embedding generation - Vector storage - Retrieval - Answer generation

It provides a high-level roadmap of how the system processes data from raw documents to final outputs.

In [ ]:
# ============================================
# BASELINE VANILLA RAG PIPELINE (Google Colab)
# Domain example: UK fiscal policy PDFs
# Copy-paste cell by cell into Colab, or run as a script.
# ============================================

# =========================


# Install Required Packages

This step installs all required libraries for building the RAG system.

pypdf → Extracts text from PDF documents
sentence-transformers → Generates embeddings
faiss → Enables fast vector similarity search
google-generativeai → Provides access to the LLM
tqdm → Displays progress bars

The -q flag ensures cleaner output during installation.

In [ ]:
# CELL 1: INSTALL PACKAGES
# =========================
!pip -q install pypdf sentence-transformers faiss-cpu google-generativeai pandas numpy scikit-learn tqdm

# =========================


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 46.7 MB/s eta 0:00:00


# Import Libraries

* os, re, json → File handling and text processing
* numpy → Numerical operations
* tqdm → Progress tracking
* pypdf → PDF parsing
* sentence-transformers → Embedding model
* faiss → Vector index
* google.generativeai → LLM interaction

These imports enable the core functionality of retrieval and generation.

In [ ]:
# CELL 2: IMPORTS
# =========================
import os
import re
import json
import textwrap
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Dict, Any, Optional

import numpy as np
import pandas as pd
from tqdm.auto import tqdm
from pypdf import PdfReader
import faiss

from sentence_transformers import SentenceTransformer
import google.generativeai as genai # Corrected import from google.genai

# =========================

# API Key Setup

This step configures the API key for accessing the generative model.

Uses Google Colab’s secure storage
Ensures authenticated access to the LLM

This is necessary for enabling the generation component of the pipeline.

In [ ]:
# CELL 3: API KEY SETUP
# =========================
from google.colab import userdata
GEMINI_API_KEY = userdata.get("GEMINI_API_KEY")

genai.configure(api_key=GEMINI_API_KEY)
GEN_MODEL = "gemini-2.5-flash" # Changed model to 'gemini-2.5-flash' to try a different model with potential higher quota

# =========================

# Upload / Locate PDFs

This step defines how input documents are loaded into the system.

Allows uploading or referencing a folder of PDFs
Identifies the dataset used for RAG

This is the data-sourcing stage: the system needs an external, specialist knowledge base.

In [ ]:
# CELL 4: UPLOAD / LOCATE PDFs
# =========================
# Option A: Use Colab upload manually
# from google.colab import files
# uploaded = files.upload()

# Option B: Put PDFs into a folder called /content/pdfs
PDF_DIR = Path("/content/pdfs") # Changed from .zip to a directory path
PDF_DIR.mkdir(parents=True, exist_ok=True)

pdf_files = sorted(PDF_DIR.glob("**/*.pdf")) # Changed to recursively find PDFs in subdirectories
assert len(pdf_files) > 0, f"No PDFs found in {PDF_DIR}. Upload your PDF files there first."
print(f"Found {len(pdf_files)} PDFs")

# =========================

Found 20 PDFs

# Unzip PDFs

If documents are compressed, this step extracts them.

Uses zipfile to unpack data
Ensures all PDFs are accessible for processing

This ensures the dataset is ready for ingestion.

In [ ]:
# CELL 4.1: UNZIP PDFs
# =========================
import zipfile

zip_path = "/content/fiscal_policy_pdfs.zip"
output_dir = "/content/pdfs"

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(output_dir)

print(f"Extracted PDFs to {output_dir}")
# =========================

Extracted PDFs to /content/pdfs

# Helper Functions

This step defines preprocessing utilities:

clean_text() → Removes noise and normalises text
May include:
Lowercasing
Removing special characters
Fixing formatting

This improves:

Embedding quality
Retrieval accuracy

In [ ]:
# CELL 5: HELPERS
# =========================
def clean_text(text: str) -> str:
    text = text.replace("\x00", " ")
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def safe_filename(x: str) -> str:
    return re.sub(r"[^a-zA-Z0-9._-]+", "_", x)

@dataclass
class Chunk:
    chunk_id: str
    source_file: str
    page_num: int
    text: str

def read_pdf_pages(pdf_path: Path) -> List[Dict[str, Any]]:
    reader = PdfReader(str(pdf_path))
    pages = []
    for i, page in enumerate(reader.pages):
        raw = page.extract_text() or ""
        txt = clean_text(raw)
        if txt:
            pages.append({
                "source_file": pdf_path.name,
                "page_num": i + 1,
                "text": txt
            })
    return pages

def make_chunks_from_pages(pages: List[Dict[str, Any]], chunk_size: int = 900, overlap: int = 150) -> List[Chunk]:
    chunks = []
    for page in pages:
        text = page["text"]
        start = 0
        idx = 0
        while start < len(text):
            end = min(len(text), start + chunk_size)
            piece = text[start:end].strip()
            if len(piece) > 120:
                chunks.append(
                    Chunk(
                        chunk_id=f'{safe_filename(page["source_file"])}_p{page["page_num"]}_c{idx}',
                        source_file=page["source_file"],
                        page_num=page["page_num"],
                        text=piece
                    )
                )
            if end == len(text):
                break
            start = end - overlap
            idx += 1
    return chunks

# =========================


# Build Chunks

This step processes documents into smaller units:

Reads each PDF
Extracts text page by page
Splits into chunks

Purpose:

Improves retrieval granularity
Prevents context overflow

This is a critical step in RAG performance.

In [ ]:
# CELL 6: BUILD CHUNKS
# =========================
all_pages = []
for pdf in tqdm(pdf_files, desc="Reading PDFs"):
    all_pages.extend(read_pdf_pages(pdf))

chunks = make_chunks_from_pages(all_pages, chunk_size=700, overlap=150)
print(f"Pages read: {len(all_pages)}")
print(f"Chunks created: {len(chunks)}")

chunk_df = pd.DataFrame([asdict(c) for c in chunks])
chunk_df.head()

# =========================


Pages read: 1061
Chunks created: 4874

,chunk_id,source_file,page_num,text
0,CC_Angels-delights-March-2026-1.pdf_p2_c0,CC_Angels-delights-March-2026-1.pdf,2,About Centre for Cities Centre for Cities is a...
1,CC_Angels-delights-March-2026-1.pdf_p2_c1,CC_Angels-delights-March-2026-1.pdf,2,breach@centreforcities.org Acknowledgements Th...
2,CC_Angels-delights-March-2026-1.pdf_p2_c2,CC_Angels-delights-March-2026-1.pdf,2,accelerate the innovation lifecycle for new pr...
3,CC_Angels-delights-March-2026-1.pdf_p3_c0,CC_Angels-delights-March-2026-1.pdf,3,1 Centre for Cities • Angels’ delights • March...
4,CC_Angels-delights-March-2026-1.pdf_p3_c1,CC_Angels-delights-March-2026-1.pdf,3,"nd institutional investment, alongside wider p..."


# Embeddings + FAISS Index

This cell creates the vector search system:

Loads embedding model (Sentence Transformers)
Converts chunks into vectors
Stores vectors in FAISS index

FAISS enables:

Fast similarity search
Scalable retrieval

This is the retrieval backbone of the system.

In [ ]:
# CELL 7: EMBEDDINGS + FAISS INDEX
# =========================
EMBED_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
embedder = SentenceTransformer(EMBED_MODEL_NAME)

chunk_texts = chunk_df["text"].tolist()
embeddings = embedder.encode(
    chunk_texts,
    batch_size=32,
    show_progress_bar=True,
    convert_to_numpy=True,
    normalize_embeddings=True
).astype("float32")

dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # cosine-like because normalized embeddings
index.add(embeddings)
print("FAISS index size:", index.ntotal)

# =========================


FAISS index size: 4874

# Baseline Retrieval Function

This defines how the system retrieves relevant information:

Converts user query into embedding
Searches FAISS index
Returns top-k similar chunks

This is a dense retrieval approach, based purely on semantic similarity.

Limitation:

No reranking
No hybrid search

In [ ]:
# CELL 8: BASELINE RETRIEVAL
# =========================
def dense_retrieve(query: str, top_k: int = 5) -> pd.DataFrame:
    q_emb = embedder.encode(
        [query],
        convert_to_numpy=True,
        normalize_embeddings=True
    ).astype("float32")
    scores, ids = index.search(q_emb, top_k)
    rows = []
    for score, idx in zip(scores[0], ids[0]):
        row = chunk_df.iloc[int(idx)].to_dict()
        row["retrieval_score"] = float(score)
        rows.append(row)
    return pd.DataFrame(rows)

# =========================


# Baseline Generation Function

This step constructs the final response:

* Combines retrieved chunks into context
* Passes context + query to LLM
* Generates answer

This is where:

Retrieval → Generation happens

Limitation:

* No advanced prompt engineering
* No verification or refinement

In [ ]:
# CELL 9: BASELINE GENERATION
# =========================
def build_context(retrieved_df: pd.DataFrame) -> str:
    parts = []
    for i, row in retrieved_df.iterrows():
        parts.append(
            f"[Source {i+1}] File: {row['source_file']} | Page: {row['page_num']}\n{row['text']}"
        )
    return "\n\n".join(parts)

def ask_gemini(prompt: str, temperature: float = 0.1) -> str:
    model = genai.GenerativeModel(GEN_MODEL)
    resp = model.generate_content(
        prompt,
        generation_config={"temperature": temperature}
    )
    return resp.text.strip()

def baseline_rag_answer(query: str, top_k: int = 5) -> Dict[str, Any]:
    retrieved = dense_retrieve(query, top_k=top_k)
    context = build_context(retrieved)

    prompt = f"""
You are a careful research assistant answering ONLY from the provided context.

Rules:
1. Use only the retrieved context below.
2. If the answer is not supported, say: "The answer is not clearly supported by the retrieved documents."
3. Be concise but specific.
4. End with a short "Sources used" line citing file names and page numbers.

Question:
{query}

Retrieved context:
{context}

Answer:
""".strip()

    answer = ask_gemini(prompt, temperature=0.1)
    return {
        "query": query,
        "answer": answer,
        "retrieved_chunks": retrieved
    }

# =========================


# Model Inspection (Optional)

This cell lists available generative models.

Helps identify which models support text generation

Used for selecting appropriate LLM.

In [ ]:
for m in genai.list_models():
  if "generateContent" in m.supported_generation_methods:
    print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gem

# Run Sample Query (Baseline Output)

This cell executes a sample query:

* Retrieves relevant chunks
* Generates response using LLM
* Displays output

This demonstrates the end-to-end functioning of the baseline RAG system.

In [ ]:
# Re-running CELL 10 with the updated generative model
query = "What are the recent concerns around UK public spending pressures?"
result = baseline_rag_answer(query, top_k=5)

print("QUESTION:\n", result["query"])
print("\nANSWER:\n", result["answer"])
display(result["retrieved_chunks"][["source_file", "page_num", "retrieval_score"]])

QUESTION:
 What are the recent concerns around UK public spending pressures?

ANSWER:
 Recent concerns around UK public spending pressures include:

*   Weak economic growth since 2008, with modest future expansion forecasts.
*   Taxes already at a historic high and rising.
*   Significantly elevated debt interest spending, with increased debt servicing costs projected to rise by £4.7 billion by 2029–30.
*   Higher-than-expected CPI inflation, leading to an additional £2.7 billion in spending, mostly from uprated social security benefits.
*   The government's commitment for defence spending to reach 3.5% of GDP by 2035, estimated to cost an additional £40 billion.
*   Substantial pressures on departmental spending, including from further industrial action in the NHS and asylum accommodation costs for the Home Office.
*   Risks related to the future costs of welfare spending following sharp growth.

Sources used:
Final_The_IFS_Green_Budget_2025_full_report_amended 2.pdf | Page: 115
Fina

,source_file,page_num,retrieval_score
0,Final_The_IFS_Green_Budget_2025_full_report_am...,115,0.694645
1,Final_The_IFS_Green_Budget_2025_full_report_am...,102,0.686066
2,Social-Market-Foundation-Publication-One-More-...,6,0.675046
3,Final_The_IFS_Green_Budget_2025_full_report_am...,120,0.673196
4,economic-and-fiscal-outlook-march-2026-web-acc...,103,0.667222


# Save Artifacts

This step saves key outputs:

* Processed data
* Embeddings
* Index

Purpose:

* Reusability
* Avoid recomputation
* Supports reproducibility

In [ ]:
# CELL 11: SAVE ARTIFACTS
# =========================
ARTIFACT_DIR = Path("/content/rag_artifacts_baseline")
ARTIFACT_DIR.mkdir(exist_ok=True)

chunk_df.to_csv(ARTIFACT_DIR / "chunks.csv", index=False)
np.save(ARTIFACT_DIR / "embeddings.npy", embeddings)
faiss.write_index(index, str(ARTIFACT_DIR / "faiss.index"))

print("Saved baseline artifacts to:", ARTIFACT_DIR)

# =========================


Saved baseline artifacts to: /content/rag_artifacts_baseline

# Evaluation Template

This provides a framework for testing:

* Defines sample queries
* Enables comparison between outputs

This supports:

* Structured evaluation
* Baseline vs enhanced comparison

> **Note:** this cell is an evaluation template. The `gold_files` values are placeholder file names, not the real corpus files, so the hit rate of 0.0 is not a real result. The baseline-vs-enhanced comparison was done by manual review against expected-answer criteria (see the [project README](README.md)).

In [ ]:
# CELL 12: SIMPLE EVALUATION TEMPLATE
# =========================
# Build a small test set manually.
# "gold_answer" is for human comparison.
# "gold_source_files" is used for rough retrieval hit-rate evaluation.
test_queries = [
    {
        "query": "What are the main risks to the UK fiscal outlook?",
        "gold_answer": "Should mention debt pressures, spending pressures, economic uncertainty, and/or borrowing risks.",
        "gold_source_files": ["obr_fiscal_outlook.pdf"]
    },
    {
        "query": "How do think tanks differ in their view on tax policy?",
        "gold_answer": "Should compare viewpoints rather than giving one isolated claim.",
        "gold_source_files": ["ifs_report.pdf", "ippr_report.pdf"]
    }
]

def evaluate_retrieval_hit_rate(test_queries: List[Dict[str, Any]], top_k: int = 5) -> pd.DataFrame:
    rows = []
    for item in test_queries:
        retrieved = dense_retrieve(item["query"], top_k=top_k)
        retrieved_files = set(retrieved["source_file"].tolist())
        gold_files = set(item.get("gold_source_files", []))
        hit = int(len(retrieved_files.intersection(gold_files)) > 0)
        rows.append({
            "query": item["query"],
            "gold_files": list(gold_files),
            "retrieved_files": list(retrieved_files),
            "hit@{}".format(top_k): hit
        })
    return pd.DataFrame(rows)

eval_df = evaluate_retrieval_hit_rate(test_queries, top_k=5)
display(eval_df)
print("Average hit rate:", eval_df["hit@5"].mean())


,query,gold_files,retrieved_files,hit@5
0,What are the main risks to the UK fiscal outlook?,[obr_fiscal_outlook.pdf],[Final_The_IFS_Green_Budget_2025_full_report_a...,0
1,How do think tanks differ in their view on tax...,"[ifs_report.pdf, ippr_report.pdf]",[Final_The_IFS_Green_Budget_2025_full_report_a...,0


Average hit rate: 0.0

# BUILD GRADIO INTERFACE

This creates an interactive chatbot:

* Takes user query
* Runs retrieval + generation
* Displays response

This demonstrates:

* Practical usability
* Real-world deployment potential

In [ ]:
# Install Gradio if not already installed
!pip -q install gradio

import gradio as gr

def rag_interface(query: str) -> str:
    """Wrapper function to call the RAG pipeline and format the output for Gradio."""
    if not query:
        return "Please enter a query."

    result = baseline_rag_answer(query, top_k=5)
    answer = result["answer"]
    retrieved_chunks = result["retrieved_chunks"]

    sources_info = []
    for _, row in retrieved_chunks.iterrows():
        sources_info.append(f"File: {row['source_file']} | Page: {row['page_num']}")

    formatted_answer = f"**Answer:**\n{answer}\n\n**Sources Used:**\n" + "\n".join(sources_info)
    return formatted_answer

# Create the Gradio interface
iface = gr.Interface(
    fn=rag_interface,
    inputs=gr.Textbox(lines=2, placeholder="Enter your query here..."),
    outputs=gr.Markdown(),
    title="Baseline Vanilla RAG Pipeline Chatbot",
    description="Ask questions about the UK fiscal policy PDFs and get answers with sources."
)

# Launch the interface
iface.launch(debug=True, share=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().


Keyboard interruption in main thread... closing server.